In [ ]:
import torch
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image
from cnn import CNN
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Ścieżka do modelu i danych testowych
model_path = "best_model.pth"  # Ścieżka do wytrenowanego modelu
data_test_path = "data/chestmnist_test_64x64.npz"  # Plik z danymi testowymi (obrazy i etykiety)

# Przygotowanie transformacji obrazów
transform = Compose([
    Resize((64, 64)),  # Dopasowanie rozmiaru obrazu
    ToTensor(),        # Konwersja na tensor
    Normalize(mean=[0.5], std=[0.5])  # Normalizacja
])

# Funkcja do wczytania danych testowych
def load_test_data(npz_path):
    data = np.load(npz_path)
    images = data['images']  # Zakładamy klucz 'images' dla obrazów
    labels = data['labels']  # Zakładamy klucz 'labels' dla etykiet
    return images, labels

# Funkcja do przetworzenia obrazu
def preprocess_image(image):
    image = Image.fromarray(image).convert("L")  # Wczytaj obraz w skali szarości
    return transform(image).unsqueeze(0)  # Dodaj wymiar batch

# Inicjalizacja modelu i wczytanie stanu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN(num_classes=14).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()  # Ustaw model w tryb ewaluacji

# Wczytaj dane testowe
images, ground_truth = load_test_data(data_test_path)

# Przechowywanie predykcji i etykiet
predictions = []
labels = []

# Testowanie modelu na danych testowych
for i in range(10):
    image = preprocess_image(images[i]).to(device)
    label = ground_truth[i]

    # Predykcja modelu
    with torch.no_grad():
        output = torch.sigmoid(model(image))
    
    # Klasyfikacja: 1 = chory, 0 = zdrowy
    is_sick = (output > 0.5).any().item()
    predictions.append(1 if is_sick else 0)
    labels.append(1 if np.any(label) else 0)  # Jeśli którakolwiek etykieta to 1, obraz jest chory

# Obliczanie metryk
accuracy = accuracy_score(labels, predictions)
precision = precision_score(labels, predictions, zero_division=1)
recall = recall_score(labels, predictions, zero_division=1)
f1 = f1_score(labels, predictions, zero_division=1)

# Wyświetl metryki
print(f"Dokładność (Accuracy): {accuracy:.2f}")
print(f"Precyzja (Precision): {precision:.2f}")
print(f"Czułość (Recall): {recall:.2f}")
print(f"F1-Score: {f1:.2f}")


C:\Users\mkowalik\AppData\Local\Temp\ipykernel_15216\3081168675.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Dokładność (Accuracy): 0.60
Precyzja (Precision): 1.00
Czułość (Recall): 0.00
F1-Score: 0.00
